# 🐼 Jour 5 — PANDAS : la boîte à outils des données

Pandas est LA bibliothèque standard de manipulation de données en Python — le cœur de la data science, du ML, de la finance… Imagine **Excel sous stéroïdes, piloté par du code** : charger, nettoyer, filtrer, agréger des tableaux de millions de lignes en quelques lignes de Python.

## 🎯 Ce que tu vas apprendre
1. Les 2 structures : **Series** (1D) et **DataFrame** (2D)
2. Inspecter un tableau : `shape`, `head`, `info`, `describe`
3. Accéder aux données : colonnes, `iloc` (par position), `loc` (par étiquette), filtres booléens
4. Créer/modifier des colonnes, agréger (`mean`, `sum`), grouper (`groupby`)
5. Gérer les valeurs manquantes (`NaN`)
6. Un exercice final complet : le parc d'attractions galactique 🚀

## 1. Les deux structures fondamentales

| Structure | Dimensions | Analogie Excel |
|---|---|---|
| **Series** | 1D (une colonne + un index) | UNE colonne de feuille |
| **DataFrame** | 2D (lignes × colonnes) | la feuille ENTIÈRE |

Un DataFrame n'est qu'un assemblage de Series (une par colonne) partageant le même index. Extraire une colonne d'un DataFrame → tu récupères une Series.

### 🔍 Démo — créer un DataFrame depuis un dictionnaire
Le dict `employee_data` : chaque CLÉ devient un NOM DE COLONNE, chaque liste devient le CONTENU de la colonne (les listes doivent avoir la même longueur !). `pd.DataFrame(dict)` assemble le tout.
💡 `display(df)` (spécifique à Jupyter) affiche le tableau en joli HTML interactif — plus lisible que `print(df)`.

In [1]:
# The standard alias for pandas used across the globe is 'pd'
import pandas as pd

# 1. Creating a DataFrame from a dictionary of lists
employee_data = {
    "Name": ["Alice", "Bob", "Charlie", "Diana"],
    "Department": ["HR", "Engineering", "Engineering", "Marketing"],
    "Salary": [65000, 85000, 92000, 72000],
    "Years_Exp": [3, 5, 7, 2]
}

df = pd.DataFrame(employee_data)

# 2. Inspecting the DataFrame
print("--- Displaying the DataFrame ---")
display(df)  # In Jupyter notebooks, display() renders interactive HTML tables!

--- Displaying the DataFrame ---


,Name,Department,Salary,Years_Exp
0,Alice,HR,65000,3
1,Bob,Engineering,85000,5
2,Charlie,Engineering,92000,7
3,Diana,Marketing,72000,2


## 2. Inspecter un DataFrame : le kit de survie

| Commande | Ce qu'elle montre |
|---|---|
| `df.shape` | (nb lignes, nb colonnes) — LA première chose à regarder |
| `df.head(n)` / `df.tail(n)` | les n premières / dernières lignes (5 par défaut) |
| `df.dtypes` | le type de chaque colonne (int64, float64, str/object…) |
| `df.info()` | le récap technique : colonnes, valeurs non-nulles (→ détecte les trous !), mémoire |
| `df.describe()` | les statistiques de chaque colonne numérique (moyenne, écart-type, quartiles…) |

**Réflexe pro à chaque nouveau dataset :** `shape` → `head()` → `info()` → `describe()`. Quatre commandes, et tu sais à qui tu as affaire.

### 🔍 Démo inspection : `shape` → (4, 4) : 4 employés × 4 colonnes. `describe()` → moyennes/quartiles des colonnes NUMÉRIQUES seulement (Name et Department, textuelles, sont ignorées). `head()`/`tail()` sur 4 lignes → tout le tableau (moins de 5 lignes !).

In [2]:
print("\n--- Shape and Quick Look ---")
print(f"Shape (Rows, Columns): {df.shape}")

print("\n--- Statistical Information ---")
display(df.describe())

print("\n--- First Elements ---")
display(df.head())

print("\n--- Last Elements ---")
display(df.tail())



--- Shape and Quick Look ---
Shape (Rows, Columns): (4, 4)

--- Statistical Information ---


,Salary,Years_Exp
count,4.000000,4.000000
mean,78500.000000,4.250000
std,12233.832869,2.217356
min,65000.000000,2.000000
25%,70250.000000,2.750000
50%,78500.000000,4.000000
75%,86750.000000,5.500000
max,92000.000000,7.000000



--- First Elements ---


,Name,Department,Salary,Years_Exp
0,Alice,HR,65000,3
1,Bob,Engineering,85000,5
2,Charlie,Engineering,92000,7
3,Diana,Marketing,72000,2



--- Last Elements ---


,Name,Department,Salary,Years_Exp
0,Alice,HR,65000,3
1,Bob,Engineering,85000,5
2,Charlie,Engineering,92000,7
3,Diana,Marketing,72000,2


`df.info()` : la carte d'identité technique — 4 entrées (lignes 0 à 3), 4 colonnes, « 4 non-null » partout (= AUCUNE valeur manquante — c'est LA ligne à scruter !), et les types.

In [3]:
print("\n--- Summary of technical info ---")
df.info()


--- Summary of technical info ---
<class 'pandas.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Name        4 non-null      str  
 1   Department  4 non-null      str  
 2   Salary      4 non-null      int64
 3   Years_Exp   4 non-null      int64
dtypes: int64(2), str(2)
memory usage: 313.0 bytes


`df.dtypes` : Name/Department → str (texte), Salary/Years_Exp → int64 (entiers 64 bits). Vérifier les types évite des surprises (un nombre stocké en texte ne se calcule pas !).

In [4]:
print("\n--- Types of each Column ---")
print(df.dtypes)


--- Types of each Column ---
Name            str
Department      str
Salary        int64
Years_Exp     int64
dtype: object


## 3. Accéder aux COLONNES

- `df["Department"]` (simples crochets) → UNE colonne = une **Series**
- `df[["Name", "Department"]]` (doubles crochets = une LISTE de noms) → un **DataFrame** réduit

C'est LA distinction simple/double crochets déjà croisée — elle devient un réflexe avec la pratique.

**Une colonne → Series :** remarque l'affichage à deux colonnes (l'index à gauche, les valeurs à droite) et le `Name: Department` en bas — signature d'une Series.

In [5]:
departments = df["Department"]
print(departments)

0             HR
1    Engineering
2    Engineering
3      Marketing
Name: Department, dtype: str


**Deux colonnes (doubles crochets) → DataFrame** : affichage tableau classique.

In [6]:
name_with_department = df[["Name", "Department"]]
display(name_with_department)

,Name,Department
0,Alice,HR
1,Bob,Engineering
2,Charlie,Engineering
3,Diana,Marketing


## 4. Accéder aux LIGNES par position : `.iloc`

`.iloc[i]` (*integer location*) : la ligne numéro i, comme dans une liste Python :
- `df.iloc[0]` → la première ligne (renvoyée comme une Series : les colonnes deviennent l'index !)
- `df.iloc[0:2]` → les lignes 0 et 1 (slicing : fin EXCLUE, comme partout en Python)

`df.iloc[0]` → la ligne d'Alice, retournée comme une Series (chaque colonne devient une étiquette d'index). Un peu déroutant au début : une LIGNE extraite se présente « verticalement ».

In [7]:
first_row = df.iloc[0]
display(first_row)

Name          Alice
Department       HR
Salary        65000
Years_Exp         3
Name: 0, dtype: object

`df.iloc[0:2]` → lignes 0 et 1 (Alice, Bob) — la ligne 2 est EXCLUE, comme dans tout slicing Python.

In [ ]:
first_two_rows = df.iloc[0:2]
display(first_two_rows)
#display(first_two_rows.iloc[1])

,Name,Department,Salary,Years_Exp
0,Alice,HR,65000,3
1,Bob,Engineering,85000,5


Name                  Bob
Department    Engineering
Salary              85000
Years_Exp               5
Name: 1, dtype: object

## 5. Accéder aux lignes par VALEUR : le filtre booléen

**Le mécanisme en 2 temps (fondamental — tu l'utiliseras partout en ML !) :**
1. `df["Name"] == "Bob"` → une Series de True/False (un verdict par ligne) — le **masque**.
2. `df[masque]` → ne garde que les lignes à True.

En une ligne : `df[df["Name"] == "Bob"]`. Ça se lit de l'intérieur vers l'extérieur : « le df, restreint aux lignes où Name vaut Bob ».

**Le filtre en action :** `df[df["Name"] == "Bob"]` → un DataFrame d'UNE ligne (celle de Bob). Remarque : le résultat garde son index d'origine (1). Ce mécanisme masque→filtre est LE pattern central de pandas — assure-toi de bien le digérer avant de continuer !

In [10]:
# We want to get the row whose "Name" is "Bob".

bobs_row = df[df["Name"] == "Bob"]
display(bobs_row)

,Name,Department,Salary,Years_Exp
1,Bob,Engineering,85000,5


## 6. Le slicing bidimensionnel : `.iloc` vs `.loc`

Pour découper lignes ET colonnes d'un coup : `df.iloc[lignes, colonnes]` / `df.loc[lignes, colonnes]`.

| | `.iloc` | `.loc` |
|---|---|---|
| S'adresse par… | **POSITIONS** (entiers) | **ÉTIQUETTES** (noms) |
| Exemple | `df.iloc[0:2, 0:2]` | `df.loc[0:2, ["Salary", "Name"]]` |
| Borne de fin | **EXCLUE** (comme les listes) | **INCLUSE** ! ⚠️ |

**⚠️ Le piège vicieux :** `iloc[0:2]` → 2 lignes (0 et 1), mais `loc[0:2]` → **3 lignes** (0, 1 ET 2) ! Avec `loc`, le slice porte sur des étiquettes (qui ici SONT des nombres, d'où la confusion) et inclut la fin. Vérifie-le dans les deux démos suivantes.

**Démo `.iloc` 2D :** `iloc[0:2, 0:2]` → lignes 0-1 × colonnes 0-1 (Name, Department — les positions !). Deuxième forme : des LISTES explicites de positions `[[0,1,2,3], [0,1,2,3]]` → tout le tableau.

In [26]:
display(df.iloc[0:2, 0:2])
display(df.iloc[[0,1,2,3], [0,1,2,3]])

,Name,Department
0,Alice,HR
1,Bob,Engineering


,Name,Department,Salary,Years_Exp
0,Alice,HR,65000,3
1,Bob,Engineering,85000,5
2,Charlie,Engineering,92000,7
3,Diana,Marketing,72000,2


**Démo `.loc` 2D :** on nomme les colonnes par ÉTIQUETTE (`["Salary", "Name"]` — et l'ordre demandé est respecté !). ⚠️ Compare les deux lignes : `loc[[0,1,2], ...]` (liste) et `loc[0:2, ...]` (slice) donnent LE MÊME résultat à 3 lignes — la preuve que la borne de fin est INCLUSE avec loc !

In [28]:
display(df.loc[[0,1,2], ["Salary", "Name"]])
display(df.loc[0:2, ["Salary", "Name"]])

,Salary,Name
0,65000,Alice
1,85000,Bob
2,92000,Charlie


,Salary,Name
0,65000,Alice
1,85000,Bob
2,92000,Charlie


## 7. Ajouter / modifier une colonne

Même syntaxe que les dictionnaires :
- `df["Bonus"] = df["Salary"] * 0.10` → colonne NOUVELLE, calculée d'un coup pour TOUTES les lignes (c'est la **vectorisation** : pas de boucle for, pandas applique l'opération à toute la colonne — concis ET rapide !)
- `df["Salary"] = df["Salary"] + 2000` → colonne existante ÉCRASÉE (augmentation générale de 2000 !)

### 🔍 Démo — vérifie dans le tableau affiché : la colonne `Bonus` est apparue (10 % de l'ANCIEN salaire), et `Salary` a augmenté de 2000. ⚠️ Détail d'ordre : le Bonus a été calculé AVANT l'augmentation — s'il fallait 10 % du nouveau salaire, il faudrait inverser les deux lignes ! L'ordre des opérations compte.

In [29]:
# Adding a new column (Bonus calculated as 10% of salary)
df["Bonus"] = df["Salary"] * 0.10

# Modifying an existing column (Giving everyone a $2,000 raise)
df["Salary"] = df["Salary"] + 2000

print("--- DataFrame after Modifications ---")
display(df)

--- DataFrame after Modifications ---


,Name,Department,Salary,Years_Exp,Bonus
0,Alice,HR,67000,3,6500.0
1,Bob,Engineering,87000,5,8500.0
2,Charlie,Engineering,94000,7,9200.0
3,Diana,Marketing,74000,2,7200.0


## 8. Les agrégations : résumer une colonne en un chiffre

`.mean()` (moyenne), `.sum()` (total), et aussi `.min()`, `.max()`, `.count()`… S'appliquent à une colonne (`df["Salary"].mean()`) ou à plusieurs (`df[["a","b"]].mean()`).

⚠️ **Repère le bug dans la démo ci-dessous :** `print(df[["Salary","Years_Exp"]].mean)` — SANS parenthèses ! Sans `()`, on affiche la MÉTHODE elle-même (`<bound method DataFrame.mean of ...>`) au lieu de l'APPELER. C'est le même piège que le `file.close` du Jour 1. Les deux lignes suivantes, avec `()`, marchent correctement.

### 🔍 Résultat — trois affichages :
1. Le `<bound method DataFrame.mean of ...>` → le BUG annoncé (`.mean` sans parenthèses) : Python affiche l'objet-méthode, pas le calcul !
2. `Average Salary: $80500` → `.mean()` AVEC parenthèses : correct (la moyenne des salaires APRÈS l'augmentation de +2000).
3. `Total Bonus: $31400` → la somme des bonus.

In [ ]:
how_many_rows = len(df)
print(df[["Salary","Years_Exp"]].mean)

mean_salary = df['Salary'].mean()
print(f"Average Salary: ${mean_salary:.2f}")

total_bonus = df['Bonus'].sum()
print(f"Total Bonus Payout: ${total_bonus:.2f}")

<bound method DataFrame.mean of    Salary  Years_Exp
0   67000          3
1   87000          5
2   94000          7
3   74000          2>
Average Salary: $80500.00
Total Bonus Payout: $31400.00


## 9. `groupby` : agréger PAR CATÉGORIE

LA question analytique type : « le salaire moyen PAR département ? ». Réponse en une ligne :
```python
df.groupby("Department")["Salary"].mean()
```
**Le mécanisme « split → apply → combine » :** 1) pandas SÉPARE les lignes par valeur de Department ; 2) calcule la moyenne de Salary DANS chaque groupe ; 3) RASSEMBLE le tout en une Series (une ligne par département). Grave ce pattern : c'est l'outil d'analyse n°1, tu le reverras dans tous les cours d'IA.

### 🔍 Résultat groupby : Engineering → 90 500 (moyenne de Bob 87k et Charlie 94k), HR → 67 000 (Alice seule), Marketing → 74 000 (Diana seule). Trois lignes de réponse pour une question analytique — c'est la puissance du groupby.

In [75]:
# Find the average salary by department
dept_salaries = df.groupby("Department")["Salary"].mean()

print("--- Average Salary by Department ---")
print(dept_salaries)

--- Average Salary by Department ---
Department
Engineering    90500.0
HR             67000.0
Marketing      74000.0
Name: Salary, dtype: float64


## 10. Les valeurs manquantes (`NaN`)

Les données réelles sont TOUJOURS trouées. Pandas marque les trous avec `NaN` (*Not a Number*).

| Outil | Rôle |
|---|---|
| `df.isna()` | tableau de True/False — où sont les trous ? |
| `df.isna().sum()` | COMPTE les trous par colonne (le réflexe n°1 !) |
| `df.fillna(valeur)` | BOUCHE les trous avec une valeur (souvent la moyenne/médiane de la colonne → « imputation ») |
| `df.dropna()` | SUPPRIME les lignes trouées (radical — on perd des données !) |

**Dans la démo :** on CASSE volontairement une case (`corrupted_data.loc[1, "Salary"] = np.nan`), puis on la répare avec la moyenne des salaires restants (`fillna({"Salary": avg})`). Remarque le `df.copy()` au début — pour ne pas abîmer l'original (le piège de mutabilité du Jour 2, version pandas !).

### 🔍 Démo réparation :
1. `corrupted_data.loc[1, "Salary"] = np.nan` → on troue la case de Bob (ligne 1, colonne Salary) — `np.nan` est LE marqueur officiel du « vide ».
2. `corrupted_data["Salary"].mean()` → la moyenne IGNORE automatiquement les NaN (calculée sur les 3 salaires restants) — comportement par défaut bien pratique.
3. `fillna({"Salary": avg_salary})` → le dict précise QUELLE colonne boucher avec QUELLE valeur. Vérifie le second tableau : Bob a reçu la moyenne des trois autres.

In [34]:
import numpy as np

# Let's introduce some missing data for demonstration
corrupted_data = df.copy()
corrupted_data.loc[1, "Salary"] = np.nan

print("--- Data with Missing Salary ---")
display(corrupted_data)

# Fix it by filling NaN with the average salary of the remaining team
avg_salary = corrupted_data["Salary"].mean()
fixed_data = corrupted_data.fillna({"Salary": avg_salary})

print("\n--- Fixed Data (NaN filled with average) ---")
display(fixed_data)

--- Data with Missing Salary ---


,Name,Department,Salary,Years_Exp,Bonus
0,Alice,HR,67000.0,3,6500.0
1,Bob,Engineering,NaN,5,8500.0
2,Charlie,Engineering,94000.0,7,9200.0
3,Diana,Marketing,74000.0,2,7200.0



--- Fixed Data (NaN filled with average) ---


,Name,Department,Salary,Years_Exp,Bonus
0,Alice,HR,67000.000000,3,6500.0
1,Bob,Engineering,78333.333333,5,8500.0
2,Charlie,Engineering,94000.000000,7,9200.0
3,Diana,Marketing,74000.000000,2,7200.0


## 11. Charger des données depuis un fichier

En vrai, on n'écrit jamais les données à la main : on les CHARGE — `pd.read_csv("fichier.csv")` → DataFrame prêt à l'emploi. (La cellule suivante est en commentaire : le chargement réel se fait dans l'exercice final.)

In [43]:
#sales_df = pd.read_csv("galaxy_park_data.csv")
#display(sales_df.head())

# 🚀 EXERCICE FINAL — Galaxy Quest Outpost

**Le scénario (traduit) :** tu es le nouveau Directeur des Données du parc d'attractions le plus couru de la Voie lactée (visiteurs humains, martiens, droïdes…). On te confie le journal des visiteurs du jour (CSV brut). Missions : **nettoyer** les données, trouver des **optimisations**, et démasquer une **menace** !

Trois sous-exercices de difficulté croissante — c'est une mini-mission de data analyst du début à la fin.

## Sous-exercice 1 — Scanner et nettoyer (facile)

**Objectif (traduit) :** inspecter et réparer :
1. charger le CSV en DataFrame ;
2. afficher le nombre exact de lignes et colonnes ;
3. compter les `Satisfaction_Score` manquants ;
4. boucher ces trous avec la moyenne générale du parc.

C'est le pipeline standard : **charger → inspecter → compter les trous → imputer.**

**Solution commentée + une bizarrerie à repérer :**
- Étapes 1-2 ✅ : chargement + `shape` → (10, 8).
- 🔍 **Étape 3 détournée :** l'énoncé demandait de COMPTER les scores manquants (`park_df["Satisfaction_Score"].isna().sum()`)… le code, lui, CRÉE un trou artificiel (`park_df.loc[1, ...] = np.nan`) sans jamais compter ! Le CSV contient déjà de vrais NaN — les compter aurait suffi.
- Étape 4 ✅ : `fillna` avec la moyenne — le pattern de la démo, correctement réappliqué.
⚠️ Effet de bord du trou artificiel : re-exécuter cette cellule re-troue puis re-bouche la ligne 1 à chaque fois — la moyenne imputée dérive légèrement à chaque exécution ! (Piège de la cellule non-idempotente, cousin du « prix divisé deux fois » des cours d'IA.)

In [60]:
# --- SUBEXERCISE 1 CODE HERE ---
# 1. Import data
# TODO
park_df = pd.read_csv("galaxy_park_data.csv")
# 2. Print shape
# TODO
print(f"Shape : {park_df.shape}")
# 3. Check for missing scores
# TODO
park_df.loc[1, "Satisfaction_Score"] = np.nan
# 4. Fill missing scores with the mean satisfaction score (in-place or re-assigned)
# TODO
satisfaction_score = park_df["Satisfaction_Score"].mean()
park_df = park_df.fillna({"Satisfaction_Score": satisfaction_score})
print("--- Subexercise 1 Cleared: Cleaned Data ---")
display(park_df)

Shape : (10, 8)
--- Subexercise 1 Cleared: Cleaned Data ---


,Visitor_ID,Visitor_Name,Alien_Species,Ticket_Type,Age,Zone_Visited,Credits_Spent,Satisfaction_Score
0,V101,Zorblax,Martian,VIP,142,Hyperspace Rollercoaster,450.5,9.2
1,V102,John Doe,Human,Standard,34,Laser Tag Arena,75.0,8.2
2,V103,BleepBloop,Droid,Standard,8,Droid Recharge Station,20.0,8.5
3,V104,Commander Shepard,Human,VIP,39,Hyperspace Rollercoaster,600.0,10.0
4,V105,Xylar,Martian,Child,12,Alien Petting Zoo,35.0,8.2
5,V106,Groot,Flora Colossus,Standard,4,Alien Petting Zoo,15.0,9.9
6,V107,GlipGlop,Martian,Standard,22,Laser Tag Arena,110.0,2.5
7,V108,Obi-Wan,Human,VIP,57,Quantum Buffet,250.0,9.5
8,V109,R2-D2,Droid,Child,3,Droid Recharge Station,0.0,7.8
9,V110,T'Challa,Human,VIP,35,Quantum Buffet,500.0,8.2


## Sous-exercice 2 — Le filtre VIP (moyen)

**Objectif (traduit) :**
1. Inflation spatiale ! Ajouter une colonne `Total_Cost_With_Tax` = `Credits_Spent` × 1.21 (21 % de taxe galactique) ;
2. extraire les « cibles à haute valeur » : les visiteurs **VIP** ayant dépensé **plus de 300 crédits** (avant taxe).

**Rappel crucial — le filtre multi-conditions :** chaque condition entre **parenthèses**, combinées par `&` (le ET de pandas — PAS le mot-clé `and`, qui plante sur les Series !) :
```python
df[(condition_1) & (condition_2)]
```

**Solution commentée :**
- Colonne taxée ✅ — mais 🔍 **coquille dans le nom** : `Total_Cast_With_Tax` (Cast au lieu de **Cost**) ! Le code marche (un nom de colonne n'est qu'une étiquette), mais quiconque cherchera `Total_Cost_With_Tax` plus tard ne la trouvera pas. Les fautes de frappe dans les noms de colonnes sont des bombes à retardement.
- Le filtre ✅ : `(park_df["Credits_Spent"] > 300) & (park_df["Ticket_Type"] == "VIP")` — les DEUX conditions entre parenthèses, combinées par `&`. Le résultat : les VIP gros dépensiers, exactement la demande.

In [74]:
# --- SUBEXERCISE 2 CODE HERE ---

# 1. Add 'Total_Cost_With_Tax'
# TODO
park_df["Total_Cast_With_Tax"] = park_df["Credits_Spent"]*(1.21)
display(park_df)
# 2. Filter for VIPs spending > 300 credits
vip_high_spenders = park_df[(park_df["Credits_Spent"] > 300 ) &(park_df["Ticket_Type"] == "VIP")]
print("--- Subexercise 2 Cleared: VIP High Spenders ---")
display(vip_high_spenders)

,Visitor_ID,Visitor_Name,Alien_Species,Ticket_Type,Age,Zone_Visited,Credits_Spent,Satisfaction_Score,Total_Cast_With_Tax
0,V101,Zorblax,Martian,VIP,142,Hyperspace Rollercoaster,450.5,9.2,545.105
1,V102,John Doe,Human,Standard,34,Laser Tag Arena,75.0,8.2,90.750
2,V103,BleepBloop,Droid,Standard,8,Droid Recharge Station,20.0,8.5,24.200
3,V104,Commander Shepard,Human,VIP,39,Hyperspace Rollercoaster,600.0,10.0,726.000
4,V105,Xylar,Martian,Child,12,Alien Petting Zoo,35.0,8.2,42.350
5,V106,Groot,Flora Colossus,Standard,4,Alien Petting Zoo,15.0,9.9,18.150
6,V107,GlipGlop,Martian,Standard,22,Laser Tag Arena,110.0,2.5,133.100
7,V108,Obi-Wan,Human,VIP,57,Quantum Buffet,250.0,9.5,302.500
8,V109,R2-D2,Droid,Child,3,Droid Recharge Station,0.0,7.8,0.000
9,V110,T'Challa,Human,VIP,35,Quantum Buffet,500.0,8.2,605.000


--- Subexercise 2 Cleared: VIP High Spenders ---


,Visitor_ID,Visitor_Name,Alien_Species,Ticket_Type,Age,Zone_Visited,Credits_Spent,Satisfaction_Score,Total_Cast_With_Tax
0,V101,Zorblax,Martian,VIP,142,Hyperspace Rollercoaster,450.5,9.2,545.105
3,V104,Commander Shepard,Human,VIP,39,Hyperspace Rollercoaster,600.0,10.0,726.000
9,V110,T'Challa,Human,VIP,35,Quantum Buffet,500.0,8.2,605.000


## Sous-exercice 3 — Démasquer le complot alien (difficile & fun !)

**Le scénario (traduit) :** la sécurité soupçonne qu'une espèce alien précise SABOTE les notes de satisfaction dans les zones où elle dépense anormalement peu — pour extorquer des pass gratuits au parc !

**Ta mission :**
1. `groupby("Alien_Species")` → moyenne de `Satisfaction_Score` ET de `Credits_Spent` par espèce → repère l'espèce à la satisfaction suspicieusement basse ;
2. filtre les lignes de CETTE espèce dont le score individuel est < 3.0 ;
3. extrais le(s) `Visitor_Name` et affiche l'alerte de sécurité !

C'est l'exercice le plus complet : il chaîne groupby + détection du minimum + masque multi-conditions + extraction de valeurs. Exactement le genre d'enquête qu'on mène sur des vraies données.

**Solution commentée — l'enquête pas à pas :**
1. Deux `groupby` séparés → crédits moyens et satisfaction moyenne par espèce. Lecture du résultat : les **Droids** dépensent 10 crédits (vs 356 pour les humains !) — les voilà, les suspects.
2. La détection du minimum, version détaillée : `min_value = mean.min()` → masque `== min_value` → `.index[0]` pour récupérer le NOM de l'espèce. Ça marche ✅ — 💡 mais pandas a un raccourci tout fait : `mean_satisfaction.idxmin()` renvoie directement l'étiquette du minimum (une ligne au lieu de trois) ! (Cousin du `idxmin` croisé dans les cours d'IA.)
3. Le masque final combine espèce coupable ET score < 3.0, puis `.tolist()` extrait les noms des saboteurs pour l'alerte. 🚨 Mission accomplie !

## 📝 Résumé du Jour 5 (et du cours Python !)
1. **Series** = colonne, **DataFrame** = tableau ; le kit d'inspection shape/head/info/describe.
2. **Accès** : `df["col"]`, `iloc` (positions, fin exclue), `loc` (étiquettes, fin INCLUSE !), et le filtre booléen `df[(cond1) & (cond2)]`.
3. **Colonnes calculées** vectorisées, **agrégations** (.mean/.sum), **groupby** = split→apply→combine.
4. **NaN** : isna().sum() pour compter, fillna(moyenne) pour imputer.
5. Tu as maintenant TOUTES les bases pour attaquer les cours d'IA (dossier Cours_IA) — où pandas devient l'outil quotidien !

In [ ]:
# --- SUBEXERCISE 3 CODE HERE ---

# 1. Group by Alien_Species and find the mean of Credits and Satisfaction
# TODO
mean_credits = park_df.groupby("Alien_Species")["Credits_Spent"].mean()
mean_satisfaction = park_df.groupby("Alien_Species")["Satisfaction_Score"].mean()
print(f"Credits moyen par les especes Aliens {mean_credits}")
print(f"le moyen de satisfaction est {mean_satisfaction}")
# 2. Use a boolean mask to isolate rows matching the culprit species with a Satisfaction_Score < 3.0
min_value = mean_satisfaction.min()
mask_nom_min_value = mean_satisfaction == min_value
print("mask_nom_min_value : ", mask_nom_min_value)
nom_min_value = mean_satisfaction[mask_nom_min_value].index
print("nom_min_value : ", nom_min_value[0])
print("type of mean satisfaction : ", type(mean_satisfaction))
print("min value : ", min_value)
print("nom min value : ", nom_min_value)
saboteur_rows = park_df[(park_df["Alien_Species"] == str(nom_min_value[0])) & (park_df["Satisfaction_Score"] < 3.0)]
display("saboteur_rows : ", saboteur_rows)
# 3. Extract the 'Visitor_Name' from those rows and print a security alert message
# Hint: You can use .iloc[0]["Visitor_Name"] or .values to grab the string out of the series.
# TODO
saboteur_names = saboteur_rows["Visitor_Name"].tolist()
print(f"Saboteur : {saboteur_names}")

print("\n--- MISSION COMPLETE ---")

Credits moyen par les especes Aliens Alien_Species
Droid              10.00
Flora Colossus     15.00
Human             356.25
Martian           198.50
Name: Credits_Spent, dtype: float64
